# 🔴 Solution: Causal Self-Attention（参考版）

## 📋 题目描述

**难度：** Medium

实现因果（掩码）自注意力。

与标准注意力类似，但阻止每个位置关注未来位置，这对 GPT 等自回归模型至关重要。

**签名:** `causal_attention(Q, K, V) -> Tensor`

**参数:**
- `Q` — 查询张量 (B, S, D)
- `K` — 键张量 (B, S, D)
- `V` — 值张量 (B, S, D)

**返回:** 因果掩码注意力输出 (B, S, D)

**约束:**
- 在 softmax 之前用 `-inf` 掩盖未来位置
- 位置 0 只能看到自身

**提示：** 标准注意力 + 因果遮蔽。`torch.triu(ones, diagonal=1)` 得到上三角 → softmax 前填 `-inf`。


In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION

def causal_attention(query: torch.Tensor, key: torch.Tensor, value: torch.Tensor) -> torch.Tensor:
    # Causal (Masked) Self-Attention
    # 核心区别：每个位置只能看到自己和之前的位置，不能看到未来
    # 这是自回归模型（GPT等）的关键，保证生成时的因果性

    d_k = key.size(-1)

    # ---- Step 1: 计算注意力分数 ----
    scores = query @ key.transpose(-2, -1) / math.sqrt(d_k)

    # ---- Step 2: 创建因果 mask ----
    # torch.triu(diagonal=1) 生成上三角矩阵（对角线以上为1，以下为0）
    # 例如 seq_len=4：
    # [[0, 1, 1, 1],
    #  [0, 0, 1, 1],
    #  [0, 0, 0, 1],
    #  [0, 0, 0, 0]]
    # 位置 i 只能看到 ≤ i 的位置
    seq_len = query.size(-2)
    causal_mask = torch.triu(torch.ones(seq_len, seq_len, device=query.device), diagonal=1).bool()

    # ---- Step 3: 应用 mask ----
    # 将上三角（未来位置）设为 -inf
    # softmax 后这些位置变为 0，实现因果性
    scores = scores.masked_fill(causal_mask, float('-inf'))

    # ---- Step 4: Softmax + 加权求和 ----
    attn_weights = torch.softmax(scores, dim=-1)
    return attn_weights @ value

In [ ]:
# Verify
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))

## 📝 核心思路总结

1. **因果性**：每个位置只能看到自己和之前的位置
2. **上三角 mask**：torch.triu(diagonal=1) 生成未来位置的 mask
3. **-inf → softmax → 0**：mask 的标准实现方式
4. **自回归基础**：GPT 等生成模型的核心机制